# Neural Identifier Training with Particle Filters - Differential-Drive Mobile Robot

In [1]:
import numpy as np
import plotly.graph_objects as go

In [2]:
# ============================================================
# 1) True nonlinear system (Differential-Drive Mobile Robot)
# ============================================================
def plant_dynamics(x, u, delta_v=0.0, delta_w=0.0):
    """
    Continuous dynamics for differential-drive mobile robot on low-traction terrain.
    x = [px, py, theta] (position and orientation)
    u = [v, w] (linear and angular velocity commands)
    
    The kinematic equations with disturbances:
    dpx/dt = (v + δv) * cos(θ)
    dpy/dt = (v + δv) * sin(θ)  
    dtheta/dt = w + δw
    """
    px, py, theta = x
    v, w = u
    
    # Apply disturbances (low traction effects)
    v_actual = v + delta_v
    w_actual = w + delta_w
    
    # Mobile robot kinematics
    px_dot = v_actual * np.cos(theta)
    py_dot = v_actual * np.sin(theta)
    theta_dot = w_actual
    
    return np.array([px_dot, py_dot, theta_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='laplacian', process_noise_std=1e-3):
    """
    One Euler step of the discrete plant with process noise.
    Low-traction terrain causes random disturbances in velocity commands.
    """
    # Generate traction disturbances
    if process_noise_type == 'laplacian':
        delta_v = np.random.laplace(0, process_noise_std * 2)  # velocity disturbance
        delta_w = np.random.laplace(0, process_noise_std)      # angular velocity disturbance
        state_noise = np.random.laplace(0, process_noise_std/10, size=3)  # small state noise
    elif process_noise_type == 'uniform':
        a_v = np.sqrt(3) * process_noise_std * 2
        a_w = np.sqrt(3) * process_noise_std
        delta_v = np.random.uniform(-a_v, a_v)
        delta_w = np.random.uniform(-a_w, a_w)
        a_state = np.sqrt(3) * process_noise_std / 10
        state_noise = np.random.uniform(-a_state, a_state, size=3)
    else:  # gaussian
        delta_v = np.random.normal(0, process_noise_std * 2)
        delta_w = np.random.normal(0, process_noise_std)
        state_noise = np.random.normal(0, process_noise_std/10, size=3)

    x_dot = plant_dynamics(x_k, u_k, delta_v, delta_w)
    x_kp1 = x_k + dt * x_dot

    return x_kp1 + state_noise

In [3]:
# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_cmd):
    """
    Features for a 3-state mobile robot system with 2 inputs:
    z = [S(px), S(py), S(θ), S(v), S(w), S(px)S(py), S(px)S(θ), S(py)S(θ), 
         S(px)S(v), S(py)S(v), S(θ)S(v), S(θ)S(w), S(v)S(w),
         S(px)^2, S(py)^2, S(θ)^2, S(v)^2, S(w)^2, 1]
    """
    s_px = sigmoidal(x_est[0])    # position x
    s_py = sigmoidal(x_est[1])    # position y  
    s_theta = sigmoidal(x_est[2]) # orientation
    s_v = sigmoidal(u_cmd[0])     # linear velocity command
    s_w = sigmoidal(u_cmd[1])     # angular velocity command
    
    return np.array([
        s_px, s_py, s_theta, s_v, s_w,                           # Linear terms
        s_px*s_py, s_px*s_theta, s_py*s_theta,                   # State cross terms
        s_px*s_v, s_py*s_v, s_theta*s_v, s_theta*s_w, s_v*s_w,  # State-input cross terms
        s_px**2, s_py**2, s_theta**2, s_v**2, s_w**2,           # Quadratic terms
        1.0                                                       # Bias
    ])


def RHONN_predict(x_state_for_z, u_cmd, w_neuron):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k), u(k) )
    """
    z_i = construct_z_vector(x_state_for_z, u_cmd)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

In [4]:
# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, u_k, x_hat_previous):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        u_k    : np.array, control inputs at time k          (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # px (measured output for mobile robot)
        x_state_for_z[1] = chi_k[1]  # py (measured output for mobile robot)

        z_i = construct_z_vector(x_state_for_z, u_k)          # shape (num_features,)
        H_i = z_i.reshape(-1, 1)                              # column vector

        for i in range(self.num_neurons):
            # Predict covariance
            P_pred = self.P[i] + self.Q[i]

            # Innovation covariance (scalar)
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-12:
                M_i = 1e-12

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update
            self.weights[i] += self.eta * K_i * e_i

            # Covariance update (Joseph or simple; we symmetrize to keep numerical hygiene)
            P_update = P_pred - np.outer(K_i, (H_i.T @ P_pred).ravel())
            self.P[i] = 0.5 * (P_update + P_update.T)  # enforce symmetry

In [5]:
# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=0.05, R_std=0.1, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        self.Q_std = Q_std
        self.R_std = R_std
        self.R_var = R_std**2
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, u_k, x_hat_previous):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : measured true states at k   (for z)
        u_k    : control inputs at k         (for z)
        x_hat_previous: previous estimate at k (to complete z)
        """
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # px (measured output for mobile robot)
        x_state_for_z[1] = chi_k[1]  # py (measured output for mobile robot)
        z = construct_z_vector(x_state_for_z, u_k)  # (num_features,)

        # 1) Predict: random walk on weights
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std

        # 2) Update: importance weights with Gaussian likelihood
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Log-likelihood for stability
            ll = -0.5 * (innov**2) / self.R_var
            ll -= np.max(ll)
            like = np.exp(ll)

            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]


In [6]:
# ============================================================
# 5) Simulation
# ============================================================
if __name__ == "__main__":
    # --- Simulation settings ---
    n_steps = 1000
    dt = 0.01
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'laplacian'  # 'laplacian' | 'uniform' | 'gaussian'
    process_noise_std = 0.05  # Higher noise for low-traction terrain

    # --- True system init ---
    x_true = np.zeros((n_steps, 3))
    x_true[0] = [0.0, 0.0, 0.0]  # Initial position and orientation [px, py, theta]
    
    # Control trajectory - circular motion with varying speed
    u_history = np.zeros((n_steps, 2))
    for k in range(n_steps):
        t = k * dt
        # Varying circular motion: v varies, w creates circular path
        u_history[k, 0] = 0.5 + 0.3 * np.sin(0.5 * t)  # Linear velocity [m/s]
        u_history[k, 1] = 0.3 + 0.2 * np.cos(0.3 * t)  # Angular velocity [rad/s]

    # --- RHONN config ---
    num_neurons = 3  # Three states for mobile robot: px, py, theta
    num_features = 19  # Updated feature vector size for 3 states + 2 inputs
    num_weights_per_neuron = num_features

    # --- Common initial weights for fair comparison ---
    # np.random.seed(12345)  # (optional) reproducibility of initial weights
    common_initial_weights = [np.random.uniform(-0.5, 0.5, num_weights_per_neuron) for _ in range(num_neurons)]
    print("Common Initial Weights:")
    for i, w in enumerate(common_initial_weights):
        print(f"  Neuron {i}: {w}")

    # --- EKF ---
    ekf_trainer = EKF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        initial_weights=common_initial_weights,
        Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.1
    )
    x_hat_ekf = np.zeros((n_steps, 3))
    x_hat_ekf[0] = x_true[0]

    # --- PF ---
    n_particles = 800
    pf_trainer = PF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        n_particles=n_particles,
        initial_weights=common_initial_weights,
        Q_std=0.8, R_std=np.sqrt(0.001), ess_threshold=n_particles / 2  # ESS < N/2
    )

    # Force identical particle initialization if desired:
    def initialize_pf_with_common_weights(pf_trainer_instance, common_weights_list):
        for i in range(pf_trainer_instance.num_neurons):
            pf_trainer_instance.particles[i] = np.tile(
                common_weights_list[i], (pf_trainer_instance.n_particles, 1)
            )
            pf_trainer_instance.weights_pf[i] = np.ones(pf_trainer_instance.n_particles) / pf_trainer_instance.n_particles

    initialize_pf_with_common_weights(pf_trainer, common_initial_weights)

    x_hat_pf = np.zeros((n_steps, 3))
    x_hat_pf[0] = x_true[0]

    print("Starting simulation...")
    for k in range(n_steps - 1):
        # ---- 1) true system -> k+1 ----
        x_true[k+1] = plant(x_true[k], u_history[k], dt, process_noise_type, process_noise_std)

        # ---- 2) EKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
        ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], u_k=u_history[k], x_hat_previous=x_hat_ekf[k])

        x_state_for_z_ekf = np.copy(x_hat_ekf[k])
        x_state_for_z_ekf[0] = x_true[k][0]  # series-parallel uses measured px at k
        x_state_for_z_ekf[1] = x_true[k][1]  # series-parallel uses measured py at k
        x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, u_history[k], ekf_trainer.weights[0])  # px
        x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, u_history[k], ekf_trainer.weights[1])  # py
        x_hat_ekf[k+1, 2] = RHONN_predict(x_state_for_z_ekf, u_history[k], ekf_trainer.weights[2])  # theta

        # ---- 3) PF update (chi_{k+1} vs z from k), then predict x_hat_{k+1} with mean weights ----
        pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], u_k=u_history[k], x_hat_previous=x_hat_pf[k])

        pf_weight_estimates = pf_trainer.get_estimate()
        x_state_for_z_pf = np.copy(x_hat_pf[k])
        x_state_for_z_pf[0] = x_true[k][0]  # series-parallel uses measured px at k
        x_state_for_z_pf[1] = x_true[k][1]  # series-parallel uses measured py at k
        x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, u_history[k], pf_weight_estimates[0])   # px
        x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, u_history[k], pf_weight_estimates[1])   # py
        x_hat_pf[k+1, 2] = RHONN_predict(x_state_for_z_pf, u_history[k], pf_weight_estimates[2])   # theta

        if k % (n_steps // 10) == 0:
            print(f"Simulation progress: {k/n_steps*100:.1f}%")

    print("Simulation finished.")

Common Initial Weights:
  Neuron 0: [-0.14905921 -0.16400213  0.06262944 -0.26565185 -0.29497121 -0.4262534
  0.0491615   0.20434457  0.22875503  0.29399572 -0.09510584 -0.22714936
 -0.32495404 -0.33427073 -0.19563757  0.21442647 -0.08053535  0.41581268
 -0.10495582]
  Neuron 1: [ 2.20976386e-01 -4.94970215e-01  4.68342965e-01 -1.52645287e-01
  1.97865705e-01 -2.45335720e-01 -4.30273623e-01 -2.55096896e-01
 -4.29126867e-01  2.17247790e-01 -4.41176943e-01 -2.65698065e-01
 -4.52498841e-04 -1.25085548e-01 -9.58404203e-02 -1.78528265e-01
 -7.18233141e-03 -3.10727101e-01 -2.65990900e-01]
  Neuron 2: [-0.38075818  0.04406759  0.29144151  0.34340322  0.20738875  0.13695482
  0.34430899 -0.14306977  0.05255892  0.10877533  0.4667749  -0.01468808
 -0.45806169 -0.44822186  0.32075104  0.39982214 -0.19298211 -0.24222044
  0.47499012]
Starting simulation...
Simulation progress: 0.0%
Simulation progress: 10.0%
Simulation progress: 10.0%
Simulation progress: 20.0%
Simulation progress: 20.0%
Simulati

In [7]:
 # ============================================================
    # 6) Results & plots for Mobile Robot System
# ============================================================
mse_px_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)  # position x
mse_py_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)  # position y
mse_theta_ekf = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2)  # orientation

mse_px_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)   # position x
mse_py_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)   # position y
mse_theta_pf = np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2)   # orientation

print(f"\nFinal EKF-RHONN Weights:")
for i in range(3):
    state_names = ['px', 'py', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {ekf_trainer.weights[i]}")

print(f"\nFinal PF-RHONN Weight Estimates:")
pf_estimates = pf_trainer.get_estimate()
for i in range(3):
    state_names = ['px', 'py', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {pf_estimates[i]}")

print("\n--- Performance Comparison (MSE) for Mobile Robot System ---")
print(f"EKF MSE px (x-position):     {mse_px_ekf:.6f}")
print(f"EKF MSE py (y-position):     {mse_py_ekf:.6f}")
print(f"EKF MSE theta (orientation): {mse_theta_ekf:.6f}")
print(f"PF  MSE px (x-position):     {mse_px_pf:.6f}")
print(f"PF  MSE py (y-position):     {mse_py_pf:.6f}")
print(f"PF  MSE theta (orientation): {mse_theta_pf:.6f}")

states_info = [
    {'idx': 0, 'var': 'px', 'desc': 'X Position', 'y_label': 'X Position (m)',
     'chi': 'χ₁ (True px)', 'x': 'x₁ (Est. px)'},
    {'idx': 1, 'var': 'py', 'desc': 'Y Position', 'y_label': 'Y Position (m)',
     'chi': 'χ₂ (True py)', 'x': 'x₂ (Est. py)'},
    {'idx': 2, 'var': 'theta', 'desc': 'Orientation', 'y_label': 'Orientation (rad)',
     'chi': 'χ₃ (True θ)', 'x': 'x₃ (Est. θ)'}
]

for state_info in states_info:
    i = state_info['idx']
    trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines',
                            name=state_info['chi'], line=dict(color='black', width=2))
    trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines',
                        name=f"{state_info['x']} (PF)", line=dict(dash='dot'))
    trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines',
                        name=f"{state_info['x']} (EKF)", line=dict(dash='dash'))

    fig = go.Figure([trace_plant, trace_pf, trace_ekf])
    fig.update_layout(
        title=f'Mobile Robot RHONN Identification for {state_info["var"]} ({state_info["desc"]})',
        xaxis_title='Time (s)',
        yaxis_title=state_info['y_label'],
        legend=dict(x=0, y=1, orientation='h'),
        font=dict(size=12),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    fig.show()

# Errors for all three states
error_px_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
error_py_ekf = x_true[:, 1] - x_hat_ekf[:, 1]
error_theta_ekf = x_true[:, 2] - x_hat_ekf[:, 2]

error_px_pf = x_true[:, 0] - x_hat_pf[:, 0]
error_py_pf = x_true[:, 1] - x_hat_pf[:, 1]
error_theta_pf = x_true[:, 2] - x_hat_pf[:, 2]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t_history, y=error_px_ekf, mode='lines',
                        name=f'EKF Error px (MSE={mse_px_ekf:.6f})', opacity=0.7))
fig2.add_trace(go.Scatter(x=t_history, y=error_px_pf, mode='lines',
                        name=f'PF Error px (MSE={mse_px_pf:.6f})', opacity=0.7))
fig2.add_trace(go.Scatter(x=t_history, y=error_py_ekf, mode='lines',
                        name=f'EKF Error py (MSE={mse_py_ekf:.6f})', opacity=0.7))
fig2.add_trace(go.Scatter(x=t_history, y=error_py_pf, mode='lines',
                        name=f'PF Error py (MSE={mse_py_pf:.6f})', opacity=0.7))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_ekf, mode='lines',
                        name=f'EKF Error θ (MSE={mse_theta_ekf:.6f})', opacity=0.7))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_pf, mode='lines',
                        name=f'PF Error θ (MSE={mse_theta_pf:.6f})', opacity=0.7))
fig2.update_layout(
    title='Mobile Robot System Identification Errors',
    xaxis_title='Time (s)',
    yaxis_title='Error',
    legend=dict(x=0, y=1, orientation='h'),
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig2.show()

# 2D Trajectory plot
fig_traj = go.Figure()
fig_traj.add_trace(go.Scatter(x=x_true[:, 0], y=x_true[:, 1],
                            mode='lines', name='True Trajectory',
                            line=dict(color='black', width=3)))
fig_traj.add_trace(go.Scatter(x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1],
                            mode='lines', name='EKF Estimation',
                            line=dict(color='red', width=2, dash='dash')))
fig_traj.add_trace(go.Scatter(x=x_hat_pf[:, 0], y=x_hat_pf[:, 1],
                            mode='lines', name='PF Estimation',
                            line=dict(color='blue', width=2, dash='dot')))

# Add start and end markers
fig_traj.add_trace(go.Scatter(x=[x_true[0, 0]], y=[x_true[0, 1]],
                            mode='markers', name='Start',
                            marker=dict(color='green', size=10, symbol='circle')))
fig_traj.add_trace(go.Scatter(x=[x_true[-1, 0]], y=[x_true[-1, 1]],
                            mode='markers', name='End',
                            marker=dict(color='red', size=10, symbol='square')))

fig_traj.update_layout(
    title='Mobile Robot - 2D Trajectory Comparison',
    xaxis_title='X Position (m)',
    yaxis_title='Y Position (m)',
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=True
)
fig_traj.show()

# Control inputs plot
fig_ctrl = go.Figure()
fig_ctrl.add_trace(go.Scatter(x=t_history[:-1], y=u_history[:-1, 0], mode='lines',
                            name='Linear Velocity (v)', line=dict(color='blue')))
fig_ctrl.add_trace(go.Scatter(x=t_history[:-1], y=u_history[:-1, 1], mode='lines',
                            name='Angular Velocity (ω)', line=dict(color='red')))
fig_ctrl.update_layout(
    title='Control Input Commands',
    xaxis_title='Time (s)',
    yaxis_title='Velocity',
    legend=dict(x=0, y=1, orientation='h'),
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig_ctrl.show()


Final EKF-RHONN Weights:
  Neuron 1 (px): [ 0.66280369 -0.37683933 -0.03401821 -0.29353439 -0.48968321  0.24697161
  0.79468201  0.04747173  0.89176091  0.36330341  0.04783946 -0.2017143
 -0.30147025  0.77107188 -0.42437871  0.12295972  0.06153517  0.33498681
 -0.54895139]
  Neuron 2 (py): [ 0.09920956  0.58422892  0.92857633 -0.31100131  0.05923272  0.25927773
 -0.3009157   0.94066694 -0.62711168  0.74031212 -0.31251861 -0.13361173
 -0.19876672 -0.31610383  1.56820469  0.52156118 -0.22905612 -0.48931506
 -0.21810621]
  Neuron 3 (theta): [-0.4079046   0.04630867  0.82348367  0.0574293  -0.17743521  0.05086169
  0.6161616   0.40877904  0.05990836  0.03296298  0.74201482  0.20636887
 -0.69250067 -0.26718416  0.54080462  1.32836552 -0.3672493  -0.5381049
  0.03326371]

Final PF-RHONN Weight Estimates:
  Neuron 1 (px): [ 1.87033406e+01 -7.37848263e+00 -2.62719992e+01  4.12889304e+01
 -5.85501818e+00 -7.17359605e+00  2.33857914e+01 -6.90178998e+00
 -3.67013625e+00  3.21935499e+01 -2.973049